In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Passes de transpilation IA
Les passes de transpilation assistées par IA sont des passes qui remplacent directement les passes Qiskit « traditionnelles » pour certaines tâches de transpilation. Elles produisent souvent de meilleurs résultats que les algorithmes heuristiques existants (comme une profondeur et un nombre de CNOT plus faibles), tout en étant beaucoup plus rapides que des algorithmes d'optimisation tels que les solveurs de satisfaisabilité booléenne. Les passes de transpilation IA peuvent s'exécuter dans ton environnement local ou dans le cloud via le Qiskit Transpiler Service, si tu fais partie du plan IBM Quantum&reg; Premium, Flex ou On-Prem (via l'API IBM Quantum Platform).

> **Note:** Les passes de transpilation assistées par IA sont en version bêta et sont susceptibles d'évoluer.
>     Si tu as des commentaires ou si tu veux contacter l'équipe de développement, utilise ce [canal Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Les passes suivantes sont actuellement disponibles :

**Passes de routage**
 - `AIRouting` : Sélection du layout et routage de circuit

**Passes de synthèse de circuit**
 - `AICliffordSynthesis` : Synthèse de circuit Clifford
 - `AILinearFunctionSynthesis` : Synthèse de circuit à fonction linéaire
 - `AIPermutationSynthesis` : Synthèse de circuit de permutation
 - `AIPauliNetworkSynthesis` : Synthèse de circuit réseau de Pauli

Pour utiliser les passes de transpilation IA, commence par [installer le package `qiskit-ibm-transpiler`](/guides/qiskit-transpiler-service#install-transpiler-service). Consulte la [documentation API de qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) pour en savoir plus sur les différentes options disponibles.

## Exécuter les passes de transpilation IA localement ou dans le cloud
Si tu veux utiliser les passes de transpilation assistées par IA dans ton environnement local gratuitement, installe `qiskit-ibm-transpiler` avec quelques dépendances supplémentaires comme suit :

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Sans ces dépendances supplémentaires, les passes de transpilation assistées par IA s'exécutent dans le cloud via le Qiskit Transpiler Service (disponible uniquement pour les utilisateurs du plan IBM Quantum Premium, Flex ou On-Prem (via l'API IBM Quantum Platform)). Après installation des dépendances supplémentaires, le mode par défaut d'exécution des passes de transpilation assistées par IA consiste à utiliser ta machine locale.

## Passe de routage IA
La passe `AIRouting` joue à la fois le rôle d'étape de layout et d'étape de routage. Elle peut être utilisée au sein d'un `PassManager` comme suit :

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Ici, le `backend` détermine la coupling map sur laquelle effectuer le routage, le `optimization_level` (1, 2 ou 3) détermine l'effort de calcul à consacrer au processus (une valeur plus élevée donne généralement de meilleurs résultats mais prend plus de temps), et le `layout_mode` précise comment gérer la sélection du layout.
Le `layout_mode` comprend les options suivantes :

- `keep` : Respecte le layout défini par les passes de transpilation précédentes (ou utilise le layout trivial si aucun n'est défini). Il est généralement utilisé uniquement lorsque le circuit doit s'exécuter sur des qubits spécifiques de l'appareil. Il produit souvent de moins bons résultats car il offre moins de marge pour l'optimisation.
- `improve` : Utilise le layout défini par les passes de transpilation précédentes comme point de départ. C'est utile lorsque tu as une bonne estimation initiale du layout ; par exemple, pour des circuits construits de manière à suivre approximativement la coupling map de l'appareil. C'est aussi utile si tu veux essayer d'autres passes de layout spécifiques combinées avec la passe `AIRouting`.
- `optimize` : Il s'agit du mode par défaut. Il fonctionne mieux pour les circuits généraux pour lesquels tu n'as peut-être pas de bonnes estimations de layout. Ce mode ignore les sélections de layout précédentes.
- `local_mode` : Ce drapeau détermine où s'exécute la passe `AIRouting`. Si `False`, `AIRouting` s'exécute à distance via le Qiskit Transpiler Service. Si `True`, le package tente d'exécuter la passe dans ton environnement local, avec un repli vers le mode cloud si les dépendances requises ne sont pas trouvées.

## Passes de synthèse de circuit IA
Les passes de synthèse de circuit IA te permettent d'optimiser des portions de différents types de circuit ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) en les re-synthétisant. Une façon typique d'utiliser la passe de synthèse est la suivante :

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

La synthèse respecte la coupling map de l'appareil : elle peut être exécutée sans risque après d'autres passes de routage sans perturber le circuit, de sorte que le circuit global continue de respecter les contraintes de l'appareil. Par défaut, la synthèse ne remplace le sous-circuit original que si le sous-circuit synthétisé est meilleur que l'original (en vérifiant uniquement le nombre de CNOT pour le moment), mais il est possible de forcer le remplacement systématique en définissant `replace_only_if_better=False`.

Les passes de synthèse suivantes sont disponibles depuis `qiskit_ibm_transpiler.ai.synthesis` :

- *AICliffordSynthesis* : Synthèse pour les circuits [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) (blocs de portes `H`, `S` et `CX`). Actuellement jusqu'à neuf blocs de qubits.
- *AILinearFunctionSynthesis* : Synthèse pour les circuits [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) (blocs de portes `CX` et `SWAP`). Actuellement jusqu'à neuf blocs de qubits.
- *AIPermutationSynthesis* : Synthèse pour les circuits [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) (blocs de portes `SWAP`). Actuellement disponible pour des blocs de 65, 33 et 27 qubits.
- *AIPauliNetworkSynthesis* : Synthèse pour les circuits réseau de Pauli (blocs de portes `H`, `S`, `SX`, `CX`, `RX`, `RY` et `RZ`). Actuellement jusqu'à six blocs de qubits.

Nous prévoyons d'augmenter progressivement la taille des blocs pris en charge.

Toutes les passes utilisent un pool de threads pour envoyer plusieurs requêtes en parallèle. Par défaut, le nombre maximum de threads correspond au nombre de cœurs plus quatre (valeurs par défaut de l'objet Python `ThreadPoolExecutor`). Tu peux toutefois définir ta propre valeur avec l'argument `max_threads` lors de l'instanciation de la passe. Par exemple, la ligne suivante instancie la passe `AILinearFunctionSynthesis` en lui permettant d'utiliser un maximum de 20 threads.